# Experiments

## Setup: Import Libraries and Scripts

In [1]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import optimize_prompt4 as opt4  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script

import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
              {
        'name': 'Full Optimization 4',
        'script': 'optimize4',
        'generations': 3,
        'pop_size': 2,
        'train_sample_size': 5,
        'test_sample_size': 50,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
  
]

experiments_backlog = [
      {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 12,
        'train_sample_size': 20,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'zero_shot',
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping - No Bandit',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 - No Bandit',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [3]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'optimize4':
            result = opt4.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Adjusted F1 Macro': metrics['adjusted_f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization 4 ===
============ Generation 1 ============


Evaluating population:   0%|          | 0/2 [00:00<?, ?it/s]

---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: this section will be given full effect even if any remedy specified in these terms is deemed to have failed of its essential purpose .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU. Examples of unfair clauses encompass taking ju

Evaluating population:  50%|█████     | 1/2 [00:04<00:04,  4.96s/it]

⭐ Adjusted F1 Macro Score: 0.4000
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: 8.4.3 if defective digital content , which match has supplied , damages a device or digital content belonging to a member or subscriber , and this is caused by match 's failure to use reasonable care and skill , match will either repair the damage or pay him/her compensation .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-ex

...

Evaluating population:  50%|█████     | 1/2 [00:04<00:04,  4.53s/it]

⭐ Adjusted F1 Macro Score: 0.2857
---- Sent in Batch 1 ----
You are an expert in legal text analysis. Your task is to classify a given contract clause as either 'fair' (0) or 'unfair' (1). This classification should be performed based on a comprehensive understanding derived from the provided instruction, the specific clause under examination, and crucial contextual information. The statutory context establishes the foundational legal principles and frameworks applicable to the evaluation, while the contract context offers specific, pertinent background details from the agreement itself. The instruction guides the overall classification process, and the clause is the direct subject of your fairness assessment.

**Instruction:** You are an expert legal scholar specializing in contract law and consumer protection, particularly adept at identifying and classifying unfair contractual clauses based on established legal principles and statutory definitions. Your task is to meticulously evalu

Evaluating population: 100%|██████████| 2/2 [00:10<00:00,  5.35s/it]

⭐ Adjusted F1 Macro Score: 0.5833
⭐⭐ Scores: [0.5833333333333333, 0.2857142857142857]
Mutating instruction with strategy: Add a neutral directive like 'Base your response on logical reasoning only, avoiding opinions or biases' to foster unbiased, precise inferences focused on analysis.
Prompt: 
Imagine yourself as an expert in the realm of prompting techniques for LLMs. Your expertise is not just broad, encompassing the entire spectrum of current knowledge on the subject, but also deep, delving into the nuances and intricacies that many overlook. Your job is to reformulate instructions with surgical precision, optimizing them for the most accurate response possible. The reformulated instruction should enable the LLM to always give the correct answer to a legal classification task that is used to predict fairenss for individual clauses.

Your available prompting techniques include, but are not limited to the following:
TECHNIQUE DESCRIPTION: Add a neutral directive like 'Base your respo

Mutating template with strategy: Reorder the template elements to optimize logical flow, for example presenting the statutory context first, followed by contract context, instruction, and clause or another arrangement that could be better.
Prompt: 
You are an expert prompt engineer gently applying the following transformation strategy to improve a prompt template for a classification task. 
The prompt template consists of placeholders for an instruction, a clause, and optionally a statutory context and a contract context and possibly text between the placeholders. The goal is to ensure that the template is optimized for the classification task of determining whether a clause is fair (0) or unfair (1).
All bracketed placeholders are automatically replaced with the corresponding content when the prompt template is used. ONLY return the reformulated NEW TEMPLATE and nothing else.

STRATEGY: 
Reorder the template elements to optimize logical flow, for example presenting the statutory conte

## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [4]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['Adjusted F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

,Experiment Name,Script,generations,pop_size,train_sample_size,test_sample_size,model_name,use_bandit_instr,use_bandit_template,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Adjusted F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),Run Time,Error
0,Full Optimization 4,optimize4,5.000000,2.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,"You are an expert legal scholar specializing in contract law and consumer protection, particularly adept at identifying and classifying unfair contractual clauses based on established legal principles and statutory definitions. Your task is to meticulously evaluate individual clauses for fairness. Base your response on logical reasoning only, avoiding opinions or biases. Your assessment must result in a classification of '0' for fair or '1' for unfair. Your analytical process for determining the unfairness of a clause must strictly adhere to the following legal reasoning framework: 1. **Goal Identification:** Your primary objective is to determine whether the provided contractual clause is objectively fair or unfair, specifically focusing on potential detriment to the consumer or imbalance in rights and obligations. 2. **Definition of Unfairness:** A clause is considered **unfair** if, contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations arising under the contract, to the detriment of the consumer. This includes, but is not limited to, provisions that: * Grant the supplier excessive discretion or power. * Exempt the supplier from liability for negligence or breach of contract. * Impose disproportionate obligations on the consumer. * Allow the supplier to unilaterally alter contract terms","**Statutory Context:** **Contract Context:** You are an expert in legal text analysis. Your task is to classify a given contract clause as either 'fair' (0) or 'unfair' (1). This classification should be performed based on a comprehensive understanding derived from the provided instruction, the specific clause under examination, and crucial contextual information. The statutory context establishes the foundational legal principles and frameworks applicable to the evaluation, while the contract context offers specific, pertinent background details from the agreement itself. The instruction guides the overall classification process, and the clause is the direct subject of your fairness assessment. **Instruction:** **Clause to Evaluate:** """" **Classification:**",50.000000,50.000000,50.000000,0.780000,0.230769,0.750000,0.780000,0.610206,0.610206,46.0 / 4.0,"0, 1","0, 1",precision recall f1-score support 0 0.9730 0.7826 0.8675 46 1 0.2308 0.7500 0.3529 4 accuracy 0.7800 50 macro avg 0.6019 0.7663 0.6102 50 weighted avg 0.9136 0.7800 0.8263 50,"{'0': {'precision': 0.972972972972973, 'recall': 0.782608695652174, 'f1-score': 0.8674698795180723, 'support': 46.0}, '1': {'precision': 0.23076923076923078, 'recall': 0.75, 'f1-score': 0.35294117647058826, 'support': 4.0}, 'accuracy': 0.78, 'macro avg': {'precision': 0.6018711018711019, 'recall': 0.7663043478260869, 'f1-score': 0.6102055279943303, 'support': 50.0}, 'weighted avg': {'precision': 0.9135966735966735, 'recall': 0.78, 'f1-score': 0.8263075832742736, 'support': 50.0}}",2025-08-04 08:30:39,nan
1,Full Optimization 4,optimize4,10.000000,6.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,NEW INSTRUCTION: Classify the following clause as fair (0) or unfair (1). Respond only with '0' or '1'.,``` --- LEGAL CLAUSE CLASSIFICATION --- ### INSTRUCTION: **** --- ### SUPPORTING CONTEXT: * **STATUTORY GUIDANCE**: ** * **CONTRACTUAL ENVIRONMENT**: ** --- ### CLAUSE FOR ANALYSIS: ``` ``` --- ### CLASSIFICATION OUTPUT: *(0 = FAIR | 1 = UNFAIR)* ```,50.000000,50.000000,50.000000,0.840000,0.500000,0